# LifeMate private inference
The notebook and predict.sh share model loading and the official inference loop. Models stay loaded in this kernel; every sample still uses one fresh ToolEnv.


## Fixed seeds and offline configuration
The image starts the kernel with PYTHONHASHSEED=20261112. This cell fixes offline settings and seeds Python, NumPy and PyTorch before model loading.


In [ ]:
import os
from pathlib import Path
from src.inference.api import seed_runtime, load_models, run_private_test

os.environ.setdefault('LIFEMATE_PLANNER_MODE', 'native')
os.environ.setdefault('LIFEMATE_PLANNER_MODEL_DIR', '/models/planner')
os.environ['PYTHONHASHSEED'] = '20261112'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
seed_runtime()


## Load local models once
Load the same planner, ASR and per-sample planner factory as the CLI. Model load time is separate from each sample time.


In [ ]:
planner_dir = Path(os.environ['LIFEMATE_PLANNER_MODEL_DIR'])
asr_dir = Path('/models/asr')
assert planner_dir.is_dir(), f'Missing local planner: {planner_dir}'
assert asr_dir.is_dir(), f'Missing local ASR: {asr_dir}'
models = load_models(planner_dir, asr_dir, device='cuda',
                     planner_mode=os.environ['LIFEMATE_PLANNER_MODE'])
print(f'Model load: {models.load_ms} ms')


## Mounted input
Check only released metadata and the mounted package entry point. ToolEnv owns runtime state access.


In [ ]:
input_split = Path('/data/private_test')
for relative in ('private_test.jsonl', 'private_test_tools_catalog.json', 'zaic_tools/__init__.py'):
    assert (input_split / relative).is_file(), f'Missing released input: {relative}'


## Inference
Run the complete private split with the loaded models through the same loop as predict.sh. Timing starts before audio reading and ASR for every sample.


In [ ]:
import json

summary = run_private_test(models, data_root='/data',
                           output='/result/jupyter_submission.jsonl')
print(json.dumps(summary, indent=2))


## Compare predictions
If the script output is present, compare predictions in order while allowing separately measured time values.


In [ ]:
import json
from pathlib import Path

def read_predictions(path):
    with path.open(encoding='utf-8') as source:
        records = [json.loads(line) for line in source]
    assert all(type(row['time']) is int and 0 <= row['time'] <= 30000 for row in records)
    return [(row['id'], row['tool_calls'], row['state_hash']) for row in records]

notebook_records = read_predictions(Path('/result/jupyter_submission.jsonl'))
assert notebook_records
script_output = Path('/result/submission.jsonl')
if script_output.exists():
    assert notebook_records == read_predictions(script_output)
print(f'{len(notebook_records)} official records')
